In [10]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from flexcraft.data.data import DesignData
from flexcraft.files import PDBFile
from flexcraft.pipelines.tcr.utils import *
from flexcraft.sequence.aa_codes import AF2_CODE, decode, PMPNN_CODE
import anarci

In [14]:
def _data_to_pos(x):
    if hasattr(x, "to_data"):
        x = x.to_data()
        assert isinstance(x, DesignData)
    if isinstance(x, DesignData):
        x = x["atom_positions"]
    return x
def order_chains(design:DesignData, order:np.ndarray):
    parts = []
    for chain in order:
        parts.append(design["chain_index"]==chain)
    return DesignData.concatenate([design[part] for part in parts], sep_chains=False)

def convert_chains(input_design:DesignData, d:dict|None=None):
    if d is None:
        d = {}
        for x,y in zip(np.sort(np.unique(input_design["chain_index"])), range(len(np.unique(input_design["chain_index"])))):
            d[int(x)]=int(y)
    print(d)
    design = input_design.update(chain_index=np.array([d[int(x)] for x in input_design["chain_index"]]))
    return design, d

def align(x:DesignData, y:DesignData, on_chain:str|int|list[str|int]):
    
    # convert chains to concrete integer range
    y,d = convert_chains(y)
    x,_ = convert_chains(x, d=d)

    on_chain=d[on_chain]

    # bring chains of x in same order
    y_chain_order = np.unique(y["chain_index"])
    x = order_chains(x, y_chain_order)

    # get chain mask
    on_chain = np.array(list(on_chain))
    chain_mask_x = (x["chain_index"][:,None]==on_chain[None,:]).any(axis=1)
    chain_mask_y = (y["chain_index"][:,None]==on_chain[None,:]).any(axis=1)

    # convert to 3d coordinates
    x = _data_to_pos(x)
    y = _data_to_pos(y)

    #

    return x

In [27]:
imgt_mapper = {
    "acdr1":(27,38),
    "acdr2":(56, 65),
    "acdr3":(105,117),
    "bcdr1":(27,38),
    "bcdr2":(56,65),
    "bcdr3":(105,117),
    }
cdr_coords = imgt_mapper.copy()
params = {
    "imgt_mapper":imgt_mapper,
    "cdr_coords":cdr_coords,
    "tcr_chain_index":np.array([0,1]),
    "mhc_chain_index":np.array([2,3]),
}
def number_anarci(
    input_design:DesignData,
    params:dict,
    chains:List[int]=[],
    code:str=AF2_CODE,
    scheme:str="imgt",
    trim:bool=True,
    classify_all:bool = False
    )->DesignData:
    
    if not chains:
        classify_all=True
        chains = np.unique(input_design["chain_index"])

    for chain in chains:
        chain_mask = np.array(input_design["chain_index"]) == chain
        # Pass only this chain's sequence to anarci
        chain_aa = np.array(input_design["aa"])[chain_mask]
        seq = decode(chain_aa, code=code)
        numbering = anarci.number(sequence=seq, scheme=scheme)
        if numbering[0]:
            print(f"Classified chain {chain} as {numbering[-1]}.")
            chain_type = numbering[-1]
            if chain_type in ["A", "L"]:
                print(f"Setting chain {chain} to {chain_type}!")
                params["tcr_chain_index"][0] = chain
            elif chain_type in ["B", "H"]:
                print(f"Setting chain {chain} to {chain_type}!")
                params["tcr_chain_index"][1] = chain
            else:
                print(f"Unknown chain type {chain_type} of chain {chain}!")
                continue
            # Build IMGT position strings (e.g. "1", "111", "111A") then convert to int
            numbering = [f"{x[0][0]}{x[0][1].strip()}" for x in numbering[0] if x[1] != "-"]
            numbering = [int(x) if x.isnumeric() else int(x[:-1]) for x in numbering]

            residue_index = np.array(input_design["residue_index"])
            if len(residue_index[chain_mask])>len(numbering):
                # extend variable region by constant region
                if trim:
                    print(f"Trimming chain {chain} to length {len(numbering)}.")
                    index = np.arange(len(residue_index))[chain_mask]
                    start = index[0]
                    stop = index[-1]+1
                    mask = np.ones(len(residue_index), dtype=np.bool_)
                    mask[start+len(numbering):stop] = False
                    input_design = input_design[mask]
                    chain_mask = np.array(input_design["chain_index"]) == chain
                    residue_index = np.array(input_design["residue_index"])
                else:
                    numbering += np.arange(numbering[-1]+1,numbering[-1]+1+chain_mask.sum()-len(numbering)).tolist()
            residue_index[chain_mask] = numbering
            input_design = input_design.update(residue_index=np.array(residue_index))

            # update cdr coords dict
            pre = ["a","b",][chain_type=="B"or chain_type=="H"]
            chain_mask = np.array(input_design["chain_index"]) == chain
            for k in params["imgt_mapper"].keys():
                if k.startswith(pre):
                    print("Configuring cdr_coords for ", k)
                    index = np.arange(chain_mask.sum())
                    mask = (input_design["residue_index"][chain_mask][:, None]==np.arange(*params["imgt_mapper"][k])[None,:]).any(axis=1)
                    params["cdr_coords"][k] = (int(index[mask][0]), int(index[mask][-1]+1))
            
        else:
            print(f"No numbering found for chain {chain}!")
            if chain in params["tcr_chain_index"]:
                print("Sequence: ", seq)
    # check if chain indices correct
    if params["tcr_chain_index"][0]==params["tcr_chain_index"][1]:
        raise ValueError("TCR chains identical! Currently only 2 chain tcrs supported.")
    if (params["mhc_chain_index"][:,None] == params["tcr_chain_index"][None,:]).any() or classify_all:
        # fix mhc chain index to longest non-tcr chain
        chains = np.unique(input_design["chain_index"])
        # mask out tcr chains
        tcr_mask = ~(chains[:,None]==params["tcr_chain_index"][None,:]).any(axis=1)
        chains = chains[tcr_mask]
        # get chain lengths
        chain_lengths =  (input_design["chain_index"][:,None] == chains[None,:]).sum(axis=0)
        # take the n longest chain indices, where n the number of non-tcr chains -1 (for the peptide chain) 
        # take all non-tcr-chains except for smalles (peptide, hopefully)
        params["mhc_chain_index"] = np.array(
            [chains[r]
            for r in np.argsort(chain_lengths)[:-(len(chains)):-1]]
        )
        print(f"Classified chains {params['mhc_chain_index']} as MHC/antigen chains")
    return input_design

In [28]:
test_file = "../../../Data/input_data/5BS0_clean.pdb"
design = PDBFile(path=test_file).to_data()
design,d = convert_chains(design)

{0: 0, 1: 1, 2: 2, 3: 3, 4: 4}


In [29]:
design = number_anarci(design, params=params)

No numbering found for chain 0!
Sequence:  GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQKMEPRAPWIEQEGPEYWDQETRNMKAHSQTDRANLGTLRGYYNQSEDGSHTIQIMYGCDVGPDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQITKRKWEAVHAAEQRRVYLEGRCVDGLRRYLENGKETLQRTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWP
No numbering found for chain 1!
Sequence:  MIQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDWSFYLLYYTEFTPTEKDEYACRVNHVTLSQPKIVKWDRDM
No numbering found for chain 2!
Classified chain 3 as A.
Setting chain 3 to A!
Trimming chain 3 to length 114.
Configuring cdr_coords for  acdr1
Configuring cdr_coords for  acdr2
Configuring cdr_coords for  acdr3
Classified chain 4 as B.
Setting chain 4 to B!
Trimming chain 4 to length 111.
Configuring cdr_coords for  bcdr1
Configuring cdr_coords for  bcdr2
Configuring cdr_coords for  bcdr3
Classified chains [0 1] as MHC/antigen chains


In [30]:
design.save_pdb("test_file.pdb")